# Tutorial: Running 4D Whole Cell Model Simulation of a Minimal Cell on the Delta Gateway

**RESOURCES**

**Primary reference:** [Thornburg Z et. al, Bringing the genetically minimal cell to life on a computer in 4D, Cell 2026](https://www.cell.com/cell/fulltext/S0092-8674(26)00174-1)

**Code:** [Luthey-Schulten-Lab/Minimal_Cell_4DWCM](https://github.com/Luthey-Schulten-Lab/Minimal_Cell_4DWCM)

**About:** This tutorial dicusses how to setup and run a 4D Whole Cell Model for a Minimal Cell on delta gateway.

**Website:** [4D Minimal Cell](https://minimalcell4d.web.illinois.edu/home/)

**Requirements:** Jupyter Kernel: LM 2.5 , 2 NVIDA GPUs


---
## 1. Setting paths and log directory

Gateway layout: repo under **`/home/user/workspace/Minimal_Cell_4DWCM`**. Creates **`logs/`** and sets env for other cells.

In [ ]:
import os
from pathlib import Path

HOME = Path("/home/user/workspace")
REPO = HOME / "Minimal_Cell_4DWCM"
LOGDIR = REPO / "logs"
LOGDIR.mkdir(parents=True, exist_ok=True)


os.environ["TMPDIR"] = "/tmp"
os.environ["HOME"] = str(HOME)
os.environ["PATH"] = "/opt/conda/envs/lm_2.5_dev/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = "/Software/Lattice_Microbes/src/pylm" + os.pathsep + os.environ.get("PYTHONPATH", "")

print("REPO:", REPO)
print("LOGDIR:", LOGDIR)

---
### Environment check (optional)


In [ ]:
import os, sys, shutil, subprocess

print("=" * 60)
print("Environment check")
print("=" * 60)
print(f"\nPython: {sys.executable}")
print(f"Version: {sys.version.split()[0]}")


print("\n--- Python packages ---")
for mod in ("jLM", "lm"):
    try:
        __import__(mod)
        print(f"  {mod}: OK")
    except ImportError as e:
        print(f"  {mod}: MISSING — {e}")

print("\n--- Executables ---")
lmp = shutil.which("lmp") or shutil.which("lmp_OMP_GPU_Kokkos") or shutil.which("lmp_delta_kokkos")
print(f"LAMMPS:       {lmp or 'NOT FOUND'}")
for label, paths in (
    ("btree_chromo", ["/Software/btree_chromo/build/apps/btree_chromo",
                      os.path.expanduser("~/Software/btree_chromo/build/apps/btree_chromo")]),
    ("gen_sc_chain (src)", ["/Software/sc_chain_generation/src/gen_sc_chain"]),
):
    p = next((x for x in paths if os.path.isfile(x)), None)
    print(f"{label:20} {p or 'NOT FOUND'}")
print("\n" + "=" * 60)

---
## 2. Create a log file

Rename the log file in the cell to store the output of the simulation we are going to run in the next cell. For instance **`logs/run.log`**.

**Note:** Make sure you don't overwrite the log file from prevoius run. The filenames should be different for each run.

In [ ]:
import subprocess

RUN_LOG = LOGDIR / "run3.log"


---
## 3. Start the simulation

---
### Command-line arguments (`Whole_Cell_Minimal_Cell.py`)

| Variable | Shorthand | Description |
|----------|-----------|-------------|
| `--outputDir` | `-od` | Name of directory (under `Data/`) to store trajectories. |
| `--simTime` | `-t` | Biological time to simulate, in **seconds** |
| `--cudaDevices` | `-cd` | Integer index of the GPU for the Lattice Microbes RDME solver |
| `--dnaSoftwareDirectory` | `-dsd` | Directory containing the `btree_chromo/` and `sc_chain_generation/` trees (often `/Software/` on the gateway) |
| `--dnaRngSeed` | `-drs` | Integer RNG seed for the chromosome / DNA pipeline |
| `--workingDirectory` | `-wd` | Base directory of the run (default: current working directory; usually set only on clusters) |

**Example executable (gateway, background + log):**

```bash
nohup /opt/conda/envs/lm_2.5_dev/bin/python -u Whole_Cell_Minimal_Cell.py \\
  -od replicate1 -t 4 -cd 0 -drs 13 -dsd /Software/ \\
  > logs/run.log 2>&1 &
```

Edit the **`python ... Whole_Cell_Minimal_Cell.py`** line below to change the output directory **`-od`** and the biological time for the simulation **`-t`**, etc. 

**Important:** Edit the log filename keeping it same as the one we created in previous cell: logs/run1.log

In the given cell we are running a test for 4 seconds of biological simulation time. For the whole cell simulation we use 2 hours (7200 seconds) of simualtion time.

In [ ]:
%%bash
set -e
mkdir -p /home/user/workspace/Minimal_Cell_4DWCM/logs
cd /home/user/workspace/Minimal_Cell_4DWCM
export PYTHONPATH=/Software/Lattice_Microbes/src/pylm
export TMPDIR=/tmp
export HOME=/home/user/workspace
nohup /opt/conda/envs/lm_2.5_dev/bin/python -u Whole_Cell_Minimal_Cell.py \
  -od replicate3 -t 10 -cd 0 -drs 13 -dsd /Software/ \
  > logs/run3.log 2>&1 &
echo "The simulation is runnung with job ID: $!"

---
## 4. Track the progress 

The simulation for 7200 of biological time takes 3 to 4 days on A100 GPUs. We should keep tracking the log file to check the progress.

The following cell prints last 80 lines of the log file. This can be used to track the progress.


In [ ]:
if RUN_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RUN_LOG)], check=False)
else:
    print(f"No log yet: {RUN_LOG}\n")

---
## 4. Restart the simulation (optional)

Run **only if the previous simulation failed** (crash, killed, OOM, time limit, etc.). This **resumes** from `Data/{output_dir}/sim_properties.pkl` using **`Restart_Whole_Cell_Minimal_Cell.py`**

This python executable has the same input variables as the original script. This will only run if outputDir is a directory that contains all associated files with a previously run simulation from the main script. For example, if you provided the input -od replicate1 previously, giving that same argument to the restart script will take the simulation state from the replicate1 directory.

The simTime variable is how much more biological time you want to run. If you have simulated 3600 seconds and want to reach a total of 6000 seconds, you would give an input of 2400 for this variable.

**Quick check it actually failed:** run previous cell 3 (`tail logs/run.log`); if the last line shows a Python traceback / abort and `pgrep -f Whole_Cell_Minimal_Cell.py` returns nothing, the run is dead.

**Important:** Edit the **`python ... Restart_Whole_Cell_Minimal_Cell.py`** line below to change the output directory **`-od`** and the biological time for the simulation left to run **`-t`**. You can change the log file or keep it same from the previous run.

In [ ]:
import subprocess

RESTART_LOG = LOGDIR / "run4.log"


In [ ]:
%%bash
set -e
cd /home/user/workspace/Minimal_Cell_4DWCM

# Refuse to restart if the previous run is still alive
if pgrep -f "Whole_Cell_Minimal_Cell.py" > /dev/null; then
  echo "ERROR: a Whole_Cell_Minimal_Cell.py process is still running."
  pgrep -af "Whole_Cell_Minimal_Cell.py" || true
  exit 1
fi

export PYTHONPATH=/Software/Lattice_Microbes/src/pylm
export TMPDIR=/tmp
export HOME=/home/user/workspace
nohup /opt/conda/envs/lm_2.5_dev/bin/python -u Restart_Whole_Cell_Minimal_Cell.py \
  -od replicate3 -t 4 -cd 0 -drs 13 -dsd /Software/ \
  > logs/RESTART_LOG.log 2>&1 &
echo "restart: nohup started, background PID: $!"

In [ ]:

if RESTART_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RESTART_LOG)], check=False)
else:
    print(f"No log yet: {RESTART_LOG}\n")

---

## 5. Cancel the simulation jobs

Sometimes you might want to cancel simulations you created by mistake. Run the next cell to identify the PID of the simulation running that you want to kill.

In [ ]:
%%bash
echo "Running jobs"
pgrep -af "Whole_Cell_Minimal_Cell.py" || echo "(none)"


In [ ]:
import subprocess

JOB_PID = 383  # e.g. 148 — set to the integer PID to stop

if not JOB_PID:
    print("Set JOB_PID to an integer (the process id), then re-run this cell.")
else:
    r = subprocess.run(["kill", str(JOB_PID)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Sent SIGTERM to PID {JOB_PID}.")
    else:
        print(f"kill failed (code {r.returncode}): {r.stderr or r.stdout or 'no such process or permission denied'}")